In [1]:
%%capture
!pip install diffusers transformers accelerate ftfy sentencepiece

In [1]:
import torch
from diffusers import CogVideoXPipeline
pipe = CogVideoXPipeline.from_pretrained("THUDM/CogVideoX1.5-5B", torch_dtype=torch.bfloat16).to("cuda")

2025-04-02 13:47:06.950336: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743601626.959989  841593 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743601626.965170  841593 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
output_latents = pipe(
    prompt="a spinning slice of delicious new-york style cheesecake with mint and berries is placed onto a plate and then drizzled with berry sauce",
    num_videos_per_prompt=1,
    num_inference_steps=50,
    num_frames=81,
    guidance_scale=6.0,
    generator=torch.Generator("cpu").manual_seed(0x7AE),
    output_type="latent"
).frames

  0%|          | 0/50 [00:00<?, ?it/s]

In [3]:
# save to disk
output_latents_path = "test_latents_cogvideox.pth"
torch.save(output_latents.half(), output_latents_path)
# unload everything except the VAE
if "pipe" in locals(): del pipe
if "transformer" in locals(): del transformer
if "output_latents" in locals(): del output_latents
import gc
gc.collect()
torch.cuda.empty_cache()
# reload from disk

In [6]:
import torch
import torch.nn as nn
from taehv import TAEHV
from diffusers.video_processor import VideoProcessor

class DotDict(dict):
    __getattr__ = dict.__getitem__
    __setattr__ = dict.__setitem__

class TAECVXDiffusersWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        self.dtype = torch.bfloat16
        self.device = torch.device("cuda")
        self.taehv = TAEHV("taecvx_fp16.pth").to(self.device, self.dtype)
        self.config = DotDict(scaling_factor=1.0)

    def decode(self, latents, return_dict=None):
        # low-memory, set parallel=True for faster + higher memory
        return DotDict(sample=self.taehv.decode_video(latents.transpose(1, 2), parallel=False).transpose(1, 2).mul_(2).sub_(1))

@torch.no_grad()
def decode_latents_cvx(vae, latents):
    latents = latents.permute(0, 2, 1, 3, 4) # [batch_size, num_channels, num_frames, height, width]
    latents = 1 / 0.7 * latents # needed to invert the scaling factor that 1.5 bakes in to the pipeline
    frames = vae.decode(latents).sample
    return frames

video_processor = VideoProcessor()
taecvx = TAECVXDiffusersWrapper()
output_latents = torch.load(output_latents_path, weights_only=True).to(taecvx.dtype)
output = video_processor.postprocess_video(video=decode_latents_cvx(taecvx, output_latents), output_type="pil")[0]

  0%|          | 0/22 [00:00<?, ?it/s]

In [9]:
from diffusers.utils import export_to_video
from IPython.display import HTML

def display_video_embed(output, output_path="taecvx_decode.mp4"):
    export_to_video(output, output_path + ".original.mp4", fps=8)
    !ffmpeg -y -i {output_path}.original.mp4 -hide_banner -loglevel error -c:v libx264 -crf 10 -threads 0 -pix_fmt yuv420p {output_path}
    return HTML(f"<video playsinline autoplay loop muted><source src='{output_path}' type='video/mp4'></video>")

display_video_embed(output)

It is recommended to use `export_to_video` with `imageio` and `imageio-ffmpeg` as a backend. 
These libraries are not present in your environment. Attempting to use legacy OpenCV backend to export video. 
Support for the OpenCV backend will be deprecated in a future Diffusers version
